# Code for the manuscript
**A Modular Data Augmentation Framework for Low-Resource Scientific and Biomedical Topic-Controlled Summarization**

This notebook contains the full pipeline used in the manuscript, organised in four parts:

| Part | Content |
|---|---|
| 0 | Common setup: shared functions for input construction, training, generation and LLM baselines |
| 1 | Science domain (TLDR+): corpus and topic annotation, pair construction, training, evaluation, LLM baselines |
| 2 | Biomedical domain (PMC+): corpus construction, splitting and topic annotation, pair construction, training, evaluation, LLM baselines |
| 3 | Additional experiments added during revision: second evaluation encoder, size-matched random-mixing baseline, statistical tests |

**Environment.** Google Colab with a GPU runtime. API keys are read from Colab Secrets (key icon in the left panel): `WF_TOKEN` (Wikifier), `HF_KEY` (Hugging Face), `NCBI_API_KEY` (optional), and `GEMINI_API_KEY`, `MISTRAL_API_KEY`, `CEREBRAS_API_KEY`, `GROQ_API_KEY` for the LLM baselines.

**Library versions.** The evaluation and the additional experiments were run with the versions in `requirements.txt`; the models were trained earlier with the versions available on Google Colab at that time.

**Seeds.** Every configuration is run with three seeds, tagged `000`, `100` and `200` (random seeds 14, 100 and 200).

Each part can be run on its own after Part 0. Paths are set in the `CONFIG` cell at the start of each section.

# Part 0 · Common setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install transformers==5.22.1 sentencepiece==0.2.2 sentence-transformers==5.7.0 rouge-score==0.1.2 \
    bert-score==0.3.13 openai==2.54.0 google-genai==2.12.1   # versions listed in requirements.txt
import os, gc, re, ast, json, time, random
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, get_scheduler

MODEL_NAME = "t5-base"
SEED_MAP   = {"000": 14, "100": 100, "200": 200}      # seed tag -> random seed
tokenizer  = T5Tokenizer.from_pretrained(MODEL_NAME)

## 0.1 Model input and training files

In [ ]:
PROMPT_PREFIX = "summarize: "

def build_input(topic, context):
    """Build the T5 input string: the target topic followed by the (mixed) source context."""
    return f"{PROMPT_PREFIX}topic={str(topic).strip()}, context={str(context).strip()}"

def to_pth(df, topic_col, context_col, target_col, out_path, max_input_length=512, max_target_len=50, batch=512):
    """Tokenise a data frame of (topic, context, target) rows and save it as a .pth training file.

    Inputs are capped at 512 tokens (the T5 limit) and targets at 50 tokens.
    The number of inputs that reach the cap is printed as a check on the length budget.
    """
    inputs_txt  = [build_input(t, c) for t, c in zip(df[topic_col], df[context_col])]
    targets_txt = [str(s).strip() for s in df[target_col]]
    inputs, targets = [], []
    for i in range(0, len(inputs_txt), batch):
        enc = tokenizer(inputs_txt[i:i+batch], max_length=max_input_length, truncation=True)
        dec = tokenizer(targets_txt[i:i+batch], max_length=max_target_len, truncation=True)
        inputs  += [{"input_ids": a, "attention_mask": m} for a, m in zip(enc["input_ids"], enc["attention_mask"])]
        targets += [{"input_ids": a, "attention_mask": m} for a, m in zip(dec["input_ids"], dec["attention_mask"])]
    n_cut = sum(len(x["input_ids"]) >= max_input_length for x in inputs)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.save({"inputs": inputs, "targets": targets}, out_path)
    print(f"saved {out_path}: {len(inputs)} rows | inputs reaching {max_input_length} tokens: {n_cut}")

## 0.2 Fine-tuning T5-base

In [ ]:
TRAIN_CFG = dict(batch_size=8, num_epochs=10, lr=5e-6, weight_decay=0.05, warmup_ratio=0.1,
                 grad_clip=1.0, patience=1, min_delta=1e-4, dropout=0.1)

def set_seed(seed):
    """Fix the Python, NumPy and PyTorch random seeds so that a run can be repeated."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_data(path):
    """Load a .pth file written by to_pth() and return a padded TensorDataset (inputs, masks, targets)."""
    data = torch.load(path, weights_only=False)
    inputs, targets = (data if isinstance(data, tuple) else (data["inputs"], data["targets"]))
    pad = tokenizer.pad_token_id
    seq = lambda xs, k: [torch.tensor(x[k]) for x in xs]
    ii = torch.nn.utils.rnn.pad_sequence(seq(inputs, "input_ids"), batch_first=True, padding_value=pad)
    im = torch.nn.utils.rnn.pad_sequence(seq(inputs, "attention_mask"), batch_first=True, padding_value=0)
    ti = torch.nn.utils.rnn.pad_sequence(seq(targets, "input_ids"), batch_first=True, padding_value=pad)
    return TensorDataset(ii, im, ti)

def train_t5(train_pth, val_pth, out_dir, seed, cfg=TRAIN_CFG):
    """Fine-tune T5-base on one training file and keep the checkpoint with the lowest validation loss.

    Training uses cross-entropy with padding tokens masked (label -100), AdamW, a linear schedule with
    warm-up, gradient clipping and early stopping on validation loss. The same settings are used for
    every configuration, so differences between models reflect the training data only.
    Writes best_model.pth and loss_log.pth (losses and optimisation steps) to out_dir.
    """
    if os.path.exists(f"{out_dir}/best_model.pth"):
        print(out_dir, "already trained, skipped"); return
    assert torch.cuda.is_available(), "Enable a GPU runtime."
    device = torch.device("cuda"); os.makedirs(out_dir, exist_ok=True); set_seed(seed)
    train_loader = DataLoader(load_data(train_pth), batch_size=cfg["batch_size"], shuffle=True, num_workers=2,
                              generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(load_data(val_pth), batch_size=cfg["batch_size"], num_workers=2)
    config = T5Config.from_pretrained(MODEL_NAME, dropout_rate=cfg["dropout"])
    model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, config=config).to(device)
    steps = len(train_loader) * cfg["num_epochs"]
    opt = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sch = get_scheduler("linear", optimizer=opt, num_warmup_steps=int(steps * cfg["warmup_ratio"]), num_training_steps=steps)

    def run_epoch(loader, train):
        model.train(train); total = 0.0
        for ii, im, ti in tqdm(loader, leave=False):
            ii, im, ti = ii.to(device), im.to(device), ti.to(device)
            labels = ti.clone(); labels[labels == tokenizer.pad_token_id] = -100
            with torch.set_grad_enabled(train):
                loss = model(input_ids=ii, attention_mask=im, labels=labels).loss
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                opt.step(); sch.step()
            total += loss.item()
        return total / len(loader)

    best, prev, bad, log = float("inf"), None, 0, {"train": [], "val": [], "steps": []}
    for epoch in range(cfg["num_epochs"]):
        tr, va = run_epoch(train_loader, True), run_epoch(val_loader, False)
        log["train"].append(tr); log["val"].append(va); log["steps"].append((epoch + 1) * len(train_loader))
        print(f"epoch {epoch+1}: train {tr:.4f} | val {va:.4f}")
        if va < best:
            best = va; torch.save(model.state_dict(), f"{out_dir}/best_model.pth")
        torch.save(log, f"{out_dir}/loss_log.pth")
        if prev is not None and (prev - va) < cfg["min_delta"]:
            bad += 1
            if bad >= cfg["patience"]:
                print("early stopping"); break
        else:
            bad = 0
        prev = va
    print(f"done | best val loss {best:.4f} | optimisation steps {log['steps'][-1]}")
    del model, opt, sch; gc.collect(); torch.cuda.empty_cache()

## 0.3 Generating summaries

In [ ]:
def generate_summaries(ckpt, topics, contexts, references, out_csv, batch=16, max_new_tokens=56, num_beams=5):
    """Generate one summary per test row with a fine-tuned checkpoint (beam search, 5 beams).

    The reference column is the target truncated to 56 tokens, the same length limit as the generated
    summaries. Writes a CSV with the columns `prediction` and `reference`, in test-set order.
    """
    if os.path.exists(out_csv):
        print(out_csv, "exists, skipped"); return
    m = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to("cuda")
    m.load_state_dict(torch.load(ckpt, map_location="cuda")); m.eval()
    inputs = [build_input(t, c) for t, c in zip(topics, contexts)]
    refs = [tokenizer.decode(tokenizer(str(t), max_length=56, truncation=True)["input_ids"], skip_special_tokens=True)
            for t in references]
    preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(inputs), batch), desc=os.path.basename(out_csv)):
            enc = tokenizer(inputs[i:i+batch], max_length=512, truncation=True, padding=True, return_tensors="pt").to("cuda")
            out = m.generate(**enc, max_new_tokens=max_new_tokens, num_beams=num_beams)
            preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    pd.DataFrame({"prediction": preds, "reference": refs}).to_csv(out_csv, index=False)
    del m; gc.collect(); torch.cuda.empty_cache()

## 0.4 Prompted LLM baselines (zero-shot, temperature 0)

In [ ]:
from google.colab import userdata
PROVIDERS = {
    "gemini-3.1-flash-lite": {"kind": "gemini", "key": "GEMINI_API_KEY", "batch": 20, "sleep": 4.1},
    "mistral-medium-2508":   {"kind": "openai", "key": "MISTRAL_API_KEY", "base_url": "https://api.mistral.ai/v1", "batch": 20, "sleep": 3.0},
    "gpt-oss-120b":          {"kind": "openai", "key": "CEREBRAS_API_KEY", "base_url": "https://api.cerebras.ai/v1", "batch": 15, "sleep": 3.0},
    "qwen/qwen3-32b":        {"kind": "openai", "key": "GROQ_API_KEY", "base_url": "https://api.groq.com/openai/v1", "batch": 5, "sleep": 3.0},
}
OUTPUT_TOKENS_PER_ITEM, TOKEN_BUFFER, MAX_RETRIES = 80, 60, 4

In [ ]:
SYSTEM_PROMPT = ("You are an expert summarizer for academic papers in computer science, biology, and medicine. "
                 "Your task is to write a concise, topic-focused abstractive summary of 1 to 2 sentences.")
FEWSHOT = """### EXAMPLES — follow this format exactly ###

[Example A: topic is the main subject of the text]
Item 0
Topic: Catastrophic interference
Text: Incremental class learning involves sequentially learning classes in bursts of examples from the same class. This violates the assumptions that underlie methods for training standard deep neural networks, and will cause them to suffer from catastrophic forgetting. FearNet is a generative model that does not store previous examples, making it memory efficient. FearNet achieves state-of-the-art performance at incremental class learning on image (CIFAR-100, CUB-200) and audio classification (AudioSet) benchmarks.
Output:
[{"id": 0, "summary": "FearNet is a memory-efficient generative model that mitigates catastrophic interference in incremental class learning by consolidating memories without storing raw examples, achieving state-of-the-art results on CIFAR-100, CUB-200, and AudioSet."}]

[Example B: topic is tangential — text focuses on a different main subject]
Item 1
Topic: Insulin resistance
Text: This study examined the effect of a 12-week aerobic exercise programme on blood pressure in overweight adults. Participants showed significant reductions in systolic and diastolic blood pressure. Fasting glucose and HbA1c levels also improved alongside the cardiovascular benefits.
Output:
[{"id": 1, "summary": "A 12-week aerobic exercise programme produced improvements relevant to insulin resistance, including reductions in fasting glucose and HbA1c, as secondary metabolic benefits alongside blood pressure reduction."}]

### RULES (non-negotiable) ###
1. Always write a positive, content-describing summary.
2. NEVER write "This item does not discuss X", "No information about X", "This paper does not address", or any negative-existence statement.
3. If the topic is not the main subject, abstract the most relevant finding(s) related to it — do not mention the topic is absent.
4. Return ONLY a valid JSON array — no markdown fences, no preamble, no extra text.
   Format: [{"id": <int>, "summary": "<string>"}, ...]
"""

def build_batch_prompt(items):
    """Build the system and user prompts for one batch of (id, topic, text) items."""
    blocks = "\n\n".join(f"Item {i}\nTopic: {t}\nText: {c}" for i, t, c in items)
    user = ("For EACH item below, write a concise abstractive summary (1–2 sentences) focused on the given Topic, "
            "following the rules and format in the examples above.\n\n" + FEWSHOT +
            "\n\n--- ITEMS TO SUMMARISE ---\n\n" + blocks)
    return SYSTEM_PROMPT, user

In [ ]:
def make_client(cfg):
    """Create the API client of a provider (Gemini SDK or an OpenAI-compatible endpoint)."""
    key = userdata.get(cfg["key"])
    if cfg["kind"] == "gemini":
        from google import genai
        return genai.Client(api_key=key)
    from openai import OpenAI
    return OpenAI(api_key=key, base_url=cfg["base_url"])

def call(client, cfg, model_id, system, user, max_out):
    """Send one request at temperature 0, retrying with exponential back-off on rate-limit errors."""
    for attempt in range(MAX_RETRIES):
        try:
            if cfg["kind"] == "gemini":
                from google.genai import types
                r = client.models.generate_content(model=model_id, contents=user, config=types.GenerateContentConfig(
                    system_instruction=system, temperature=0, top_p=1, max_output_tokens=max_out))
                return r.text
            r = client.chat.completions.create(model=model_id, temperature=0, top_p=1, max_tokens=max_out,
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
            return r.choices[0].message.content
        except Exception as e:
            if any(s in str(e) for s in ["503", "429", "UNAVAILABLE", "rate"]):
                time.sleep(2 ** attempt + random.random())
            else:
                raise
    raise RuntimeError("max retries exceeded")

def run_llm(model_id, df, topic_col, ctx_col, out_csv, max_context_chars=None):
    """Summarise every row with one LLM in batches; raw outputs are appended to out_csv so a run can resume.
    Returns the input frame with a `prediction` column."""
    cfg = PROVIDERS[model_id]; client = make_client(cfg)
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["id", "summary"])
    done = done[done["summary"].astype(str).str.strip().ne("") & done["summary"].notna()]
    todo = [i for i in range(len(df)) if i not in set(done["id"])]
    for s in tqdm(range(0, len(todo), cfg["batch"]), desc=model_id):
        ids = todo[s:s + cfg["batch"]]
        items = [(i, str(df.iloc[i][topic_col]), str(df.iloc[i][ctx_col])[:max_context_chars]) for i in ids]
        system, user = build_batch_prompt(items)
        try:
            raw = call(client, cfg, model_id, system, user, len(ids) * OUTPUT_TOKENS_PER_ITEM + TOKEN_BUFFER)
            raw = re.sub(r"<think>.*?</think>", "", raw or "", flags=re.S)
            parsed = json.loads(re.sub(r"^```json\s*|```$", "", raw.strip(), flags=re.M).strip())
            rows = [{"id": p["id"], "summary": p["summary"]} for p in parsed if p.get("id") in ids]
            pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
        except Exception as e:
            print("batch", s, "failed:", str(e)[:200])
        time.sleep(cfg["sleep"])
    res = pd.read_csv(out_csv).drop_duplicates("id", keep="last").set_index("id")
    out = df.copy(); out["prediction"] = [res["summary"].get(i, "") for i in range(len(df))]
    print("missing:", (out["prediction"].astype(str).str.strip() == "").sum())
    return out

# Part 1 · Science domain (TLDR+)

## 1.1 Corpus and topic annotation (SciTLDR-A + Wikifier)

In [ ]:
# ===== CONFIG =====
SCI_BASE = "/content/drive/MyDrive/Main Thesis/Sci_TLDR"
WIKIFIER_URL = "http://www.wikifier.org/annotate-article"
DF_IGNORE_VAL, WORDS_IGNORE_VAL = 50, 50     # Wikifier: ignore the most frequent terms
WIKIFIER_COSINE_MIN = 0.1                    # minimum cosine between a concept and the text
SLEEP = 0.2

In [ ]:
!pip -q install datasets==3.6.0 requests
import requests
from datasets import load_dataset
os.makedirs(SCI_BASE, exist_ok=True)
WF_KEY = userdata.get("WF_TOKEN")

In [ ]:
SPECIAL_TOKENS = {"~silence~", "~SILENCE~", "~SIL", "[SILENCE]", "[hesitation]", "[HESITATION]",
                  "<unk>", "[UNKNOWN]", "[unknown]"}

def clean(text):
    """Remove noise and placeholder tokens from a SciTLDR text."""
    for t in SPECIAL_TOKENS:
        text = text.replace(t, "")
    return text.strip()

def to_rows(split):
    """Flatten one SciTLDR-A split: one row per (abstract, reference TLDR) pair."""
    rows = []
    for ex in load_dataset("allenai/scitldr", "Abstract")[split]:
        source = clean(" ".join(ex["source"]))
        for tgt in ex["target"]:
            rows.append({"paper_id": ex["paper_id"], "source": source, "target": clean(tgt)})
    return pd.DataFrame(rows)

sci = {s: to_rows(s) for s in ["train", "validation", "test"]}
for k, v in sci.items():
    print(k, len(v), "rows |", v.paper_id.nunique(), "papers")

In [ ]:
CACHE = f"{SCI_BASE}/wikifier_cache.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
session = requests.Session()

def wikify(text, retries=3):
    """Return the topic of a TLDR: the Wikifier concept with the highest PageRank among concepts
    whose cosine similarity to the text exceeds WIKIFIER_COSINE_MIN. Results are cached on Drive."""
    if text in cache:
        return cache[text]
    for k in range(retries):
        try:
            r = session.post(WIKIFIER_URL, data={"text": text, "userKey": WF_KEY,
                             "nTopDfValuesToIgnore": DF_IGNORE_VAL, "nWordsToIgnoreFromList": WORDS_IGNORE_VAL}, timeout=30)
            if r.status_code == 200:
                ann = r.json().get("annotations", [])
                cand = sorted([a for a in ann if a["cosine"] > WIKIFIER_COSINE_MIN], key=lambda a: a["pageRank"], reverse=True)
                cache[text] = cand[0]["title"] if cand else None
                return cache[text]
        except Exception:
            pass
        time.sleep(2 ** k)
    return None

for k, v in sci.items():
    v["topic"] = [wikify(t) for t in tqdm(v["target"], desc=k)]
    json.dump(cache, open(CACHE, "w"))
    time.sleep(SLEEP)
    print(k, "missing topics:", v["topic"].isna().sum())

In [ ]:
def alternative_topics(df):
    """For each reference, list the topics of the other references of the same paper that differ from
    its own topic. These are the competing topics used by the Win Rate in the science domain."""
    alts = []
    for pid, own in zip(df["paper_id"], df["topic"]):
        others = df.loc[df["paper_id"] == pid, "topic"].dropna().astype(str)
        alts.append(sorted({t for t in others if t.lower() != str(own).lower()}))
    return alts

for k, name in [("train", "train"), ("validation", "val"), ("test", "test")]:
    d = sci[k].dropna(subset=["topic"]).reset_index(drop=True)
    if k == "test":
        d["alter_topic_filtered"] = [str(a) for a in alternative_topics(d)]
    d.to_csv(f"{SCI_BASE}/Scitldr+_{name}.csv", index=False)
    print(name, len(d), "rows saved")

## 1.2 Pair construction (ψ), difficulty tiers and augmentation scales (ξ)
Pairs of training abstracts are scored by the cosine similarity of their reference TLDRs (all-MiniLM-L6-v2) and kept only if both abstracts together fit within 500 tokens. Similarity is split into three tiers. Each scale *k*X contains *k* × 2,000 pairs, drawn evenly from the three tiers; smaller scales are subsets of larger ones.

In [ ]:
# ===== CONFIG =====
POOL_SEEDS = SEED_MAP
POOL_SIZE = 100_000
MAX_PAIR_TOKENS = 500
SCI_TIERS = (0.154, 0.25)        # similarity thresholds between tiers 1/2 and 2/3
STEP = 2000                      # 1X = 2,000 pairs
SCALES = list(range(1, 17)) + [25, 30]
TIER_SET_SCALE = 7               # size of the tier-configuration sets (RQ3) = 7X

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("all-MiniLM-L6-v2")
train = pd.read_csv(f"{SCI_BASE}/Scitldr+_train.csv")
val = pd.read_csv(f"{SCI_BASE}/Scitldr+_val.csv")
print(len(train), "training rows")

def sim_and_tokens(df):
    """Return the cosine-similarity matrix of the reference summaries and the token length of each source."""
    emb = st_model.encode(df["target"].tolist(), batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    return emb @ emb.T, np.array([len(tokenizer.encode(s)) for s in df["source"]])

def rows_from_pairs(df, a, b, sim):
    """Build mixed-context rows for the pairs (a[i], b[i]): concatenated sources plus both targets and topics."""
    A, B = df.iloc[a].reset_index(drop=True), df.iloc[b].reset_index(drop=True)
    return pd.DataFrame({"combined_text": A["source"] + " " + B["source"],
                         "context1": A["source"], "context2": B["source"],
                         "target1": A["target"], "target2": B["target"],
                         "topic1": A["topic"], "topic2": B["topic"], "similarity_score": sim[a, b]})

def assign_tier(s, th=SCI_TIERS):
    """Map similarity scores to difficulty tiers 1 (low), 2 (medium) and 3 (high)."""
    return np.where(s < th[0], 1, np.where(s <= th[1], 2, 3))

SIM, TOK = sim_and_tokens(train)
ok = (TOK[:, None] + TOK[None, :] <= MAX_PAIR_TOKENS) & (SIM > 0)
np.fill_diagonal(ok, False)
CAND = np.flatnonzero(ok)
pools = {}
for tag, seed in POOL_SEEDS.items():
    pick = np.random.default_rng(seed).choice(CAND, size=min(POOL_SIZE, len(CAND)), replace=False)
    a, b = np.unravel_index(pick, ok.shape)
    p = rows_from_pairs(train, a, b, SIM)
    p["Terciles"] = assign_tier(p["similarity_score"].values)
    pools[tag] = p
    p.to_csv(f"{SCI_BASE}/pool_100k_{tag}.csv", index=False)
    print(tag, len(p), "pairs | tier %", (p.Terciles.value_counts(normalize=True).sort_index() * 100).round(1).to_dict())

In [ ]:
def swap_half(df, seed, by="Terciles"):
    """Within each tier, swap the target and topic of a random half of the pairs, so that the target
    comes from the first or the second context with equal probability."""
    df = df.copy(); rng = np.random.default_rng(seed)
    for _, g in df.groupby(by):
        idx = rng.choice(g.index.values, size=len(g) // 2, replace=False)
        df.loc[idx, ["target1", "target2"]] = df.loc[idx, ["target2", "target1"]].values
        df.loc[idx, ["topic1", "topic2"]] = df.loc[idx, ["topic2", "topic1"]].values
    return df

def nested_scales(pool, seed):
    """Build the 1X-30X training sets. Pairs are interleaved T1, T2, T3, T1, ... so that every scale is
    balanced across tiers, and each smaller scale is a prefix (subset) of the larger ones."""
    rng = np.random.default_rng(seed)
    pool = swap_half(pool.drop_duplicates(["context1", "context2"]).reset_index(drop=True), seed)
    n_max = max(SCALES) * STEP
    per_tier = -(-n_max // 3)
    tiers = {t: rng.permutation(pool.index[pool.Terciles == t].values) for t in (1, 2, 3)}
    for t, idx in tiers.items():
        if len(idx) < per_tier:
            raise ValueError(f"tier {t}: {len(idx)} pairs < {per_tier} needed for {max(SCALES)}X")
    order = []
    for i in range(per_tier):
        order += [tiers[1][i], tiers[2][i], tiers[3][i]]
    order = np.array(order[:n_max])
    out = {}
    for k in SCALES:
        d = pool.loc[order[:k * STEP]].reset_index(drop=True)
        out[f"{k}x"] = d
        print(f"{k:>2}X n={len(d):>6}", d.Terciles.value_counts().sort_index().to_dict())
    return out

scaled = {}
for tag, seed in POOL_SEEDS.items():
    scaled[tag] = nested_scales(pools[tag], seed)
    os.makedirs(f"{SCI_BASE}/Seed_{tag}", exist_ok=True)
    for k, d in scaled[tag].items():
        d.to_csv(f"{SCI_BASE}/Seed_{tag}/MixTLDR_{k}_{tag}.csv", index=False)

In [ ]:
# Tier configurations for RQ3: single tiers (T1, T2, T3) and tier pairs, each the size of 7X
TIER_SETS = {"T1": [1], "T2": [2], "T3": [3], "T1T2": [1, 2], "T1T3": [1, 3], "T2T3": [2, 3]}
for tag, seed in POOL_SEEDS.items():
    p = swap_half(pools[tag].drop_duplicates(["context1", "context2"]), seed)
    n = TIER_SET_SCALE * STEP
    os.makedirs(f"{SCI_BASE}/tiers_{tag}", exist_ok=True)
    for name, tiers in TIER_SETS.items():
        per = n // len(tiers)
        d = pd.concat([p[p.Terciles == t].sample(per, random_state=seed) for t in tiers]).sample(frac=1, random_state=seed)
        d.to_csv(f"{SCI_BASE}/tiers_{tag}/{name}_{tag}.csv", index=False)
        print(tag, name, len(d))

In [ ]:
# Validation set: papers with a single reference are paired with another abstract (mixed context);
# papers with several references are kept as single-context rows.
counts = val.paper_id.value_counts()
single_ref = val[val.paper_id.isin(counts[counts == 1].index)].reset_index(drop=True)
multi_ref = val[~val.paper_id.isin(single_ref.paper_id)]
vs, vt = sim_and_tokens(single_ref)
rng = np.random.default_rng(14); a, b = [], []
for i in range(len(single_ref)):
    valid = np.flatnonzero(vt <= MAX_PAIR_TOKENS - vt[i]); valid = valid[valid != i]
    if len(valid):
        a.append(i); b.append(rng.choice(valid))
mixed = rows_from_pairs(single_ref, np.array(a), np.array(b), vs)
val_new = pd.concat([mixed.rename(columns={"combined_text": "source", "target1": "target", "topic1": "topic"})[["source", "target", "topic"]],
                     multi_ref[["source", "target", "topic"]]], ignore_index=True)
val_new.to_csv(f"{SCI_BASE}/New_val.csv", index=False)
print(len(val_new), "validation rows")

In [ ]:
for tag in scaled:
    for k, d in scaled[tag].items():
        to_pth(d, "topic1", "combined_text", "target1", f"{SCI_BASE}/Seed_{tag}/{k}.pth")
    for name in TIER_SETS:
        d = pd.read_csv(f"{SCI_BASE}/tiers_{tag}/{name}_{tag}.csv")
        to_pth(d, "topic1", "combined_text", "target1", f"{SCI_BASE}/tiers_{tag}/{name}.pth")
to_pth(val_new, "topic", "source", "target", f"{SCI_BASE}/real_val.pth")
to_pth(train, "topic", "source", "target", f"{SCI_BASE}/baseline1_train.pth")   # Baseline1: no mixing

## 1.3 Training

In [ ]:
SCI_VAL_PTH = f"{SCI_BASE}/real_val.pth"
SCI_RUNS = {f"{k}x": lambda tag, k=k: f"{SCI_BASE}/Seed_{tag}/{k}x.pth" for k in SCALES}
SCI_RUNS.update({name: lambda tag, n=name: f"{SCI_BASE}/tiers_{tag}/{n}.pth" for name in TIER_SETS})
SCI_RUNS["baseline1"] = lambda tag: f"{SCI_BASE}/baseline1_train.pth"

for tag, seed in SEED_MAP.items():
    for name, pth in SCI_RUNS.items():
        train_t5(pth(tag), SCI_VAL_PTH, f"{SCI_BASE}/models/{name}_{tag}", seed)

## 1.4 Generating test summaries

In [ ]:
SCI_TEST_CSV = f"{SCI_BASE}/Scitldr+_test.csv"
def sci_result_path(name, seed):
    """Location of the generated summaries of one model and seed."""
    return f"{SCI_BASE}/Seed_{seed}/Mixed/Result/result{name}_{seed}.csv"

test = pd.read_csv(SCI_TEST_CSV)
for tag in SEED_MAP:
    for name in SCI_RUNS:
        generate_summaries(f"{SCI_BASE}/models/{name}_{tag}/best_model.pth", test["topic"], test["source"],
                           test["target"], sci_result_path(name, tag))

## 1.5 Evaluation
ROUGE-L, BERTScore F1 and the cosine similarity between generated and reference summaries measure summary quality. Topic alignment uses all-MiniLM-L6-v2: a summary *wins* if it is more similar to its reference topic than to every alternative topic of the same paper; Avg_ref and Avg_alt are the mean similarities to the reference and alternative topics, and Diff = Avg_ref − Avg_alt.

In [ ]:
# ===== CONFIG =====
SEEDS     = ["000", "100", "200"]
ALT_PARSE = "original"    # how alternative-topic lists are read: "original" (as reported) | "fixed" (literal list)
EMPTY_ALT = "win"         # rows without alternatives: counted as a win ("win", as reported) or excluded ("exclude")

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers.utils import logging
logging.set_verbosity_error()
st = SentenceTransformer("all-MiniLM-L6-v2")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
test = pd.read_csv(SCI_TEST_CSV)

def parse_alt(x):
    """Read the alternative-topic list stored in the test file."""
    s = "" if pd.isna(x) else str(x)
    if ALT_PARSE == "original":
        return [t.strip() for t in s.split(",")]
    try:
        return [str(t) for t in ast.literal_eval(s)]
    except Exception:
        return []

test["alt_topics"] = test["alter_topic_filtered"].apply(parse_alt)
topic_vocab = sorted({str(t) for t in test["topic"]} | {t for a in test["alt_topics"] for t in a})
TV = dict(zip(topic_vocab, st.encode(topic_vocab, convert_to_numpy=True, normalize_embeddings=True)))
print(len(test), "test rows")

In [ ]:
def score_file(preds, refs):
    """Compute summary-quality and topic-alignment metrics for one result file (rows in test-set order)."""
    preds = [str(p) if pd.notna(p) else "" for p in preds]
    refs  = [str(r) for r in refs]
    rl = np.mean([scorer.score(p, r)["rougeL"].fmeasure for p, r in zip(preds, refs)])
    _, _, f1 = bert_score(preds, refs, lang="en", verbose=False)
    pe = st.encode(preds, convert_to_numpy=True, normalize_embeddings=True)
    re_ = st.encode(refs, convert_to_numpy=True, normalize_embeddings=True)
    ref = np.array([pe[i] @ TV[str(t)] for i, t in enumerate(test["topic"])])
    win, alt = [], []
    for i, a in enumerate(test["alt_topics"]):
        if not a:
            win.append(1.0 if EMPTY_ALT == "win" else np.nan); alt.append(np.nan); continue
        s = np.array([pe[i] @ TV[t] for t in a])
        win.append(float(ref[i] > s.max())); alt.append(s.mean())
    avg_ref, avg_alt = float(ref.mean()), float(np.nanmean(alt))
    return {"rougeL": rl, "bert_f1": f1.mean().item(), "cosine_sim": float((pe * re_).sum(1).mean()),
            "wins": int(np.nansum(win)), "win_rate": 100 * np.nanmean(win),
            "avg_ref": avg_ref, "avg_alt": avg_alt, "diff": avg_ref - avg_alt}

def run_seed(seed, names=None):
    """Score every model of one seed and save the per-seed table (input for the statistical tests)."""
    rows = []
    for name in tqdm(names or [f"{n}x" for n in SCALES], desc=f"seed {seed}"):
        path = sci_result_path(name, seed)
        if not os.path.exists(path):
            print("not found:", path); continue
        d = pd.read_csv(path)
        if len(d) != len(test):
            print(f"skipped {path}: {len(d)} rows (test set {len(test)})"); continue
        rows.append({"model": name, "seed": seed, **score_file(d["prediction"], d["reference"])})
    df = pd.DataFrame(rows)
    df.to_csv(f"{SCI_BASE}/per_seed_{seed}.csv", index=False)
    return df

sci_per_seed = pd.concat([run_seed(s) for s in SEEDS], ignore_index=True)
sci_per_seed.to_csv(f"{SCI_BASE}/per_seed.csv", index=False)
sci_per_seed.groupby("model").mean(numeric_only=True).round(4)

## 1.6 Prompted LLM baselines

In [ ]:
SCI_LLM_DIR = f"{SCI_BASE}/LLM-0-shot"
os.makedirs(SCI_LLM_DIR, exist_ok=True)
test = pd.read_csv(SCI_TEST_CSV)
for model_id in PROVIDERS:
    out = run_llm(model_id, test, "topic", "source", f"{SCI_LLM_DIR}/llm_raw_{model_id.replace('/', '_')}.csv")
    out.to_csv(f"{SCI_LLM_DIR}/llm_{model_id.split('/')[-1]}.csv", index=False)

In [ ]:
# Score the LLM outputs with the same metrics (reference = full target)
llm_rows = []
for model_id in PROVIDERS:
    d = pd.read_csv(f"{SCI_LLM_DIR}/llm_{model_id.split('/')[-1]}.csv")
    llm_rows.append({"model": model_id, **score_file(d["prediction"], d["target"])})
pd.DataFrame(llm_rows).round(4)

# Part 2 · Biomedical domain (PMC+)

## 2.1 Corpus construction
PubMed Central articles published in 2015 under CC BY or CC0 licences are fetched through the NCBI E-utilities. The abstract provides the targets (its Background and Conclusion sections) and the full text provides the source contexts (its Introduction and Conclusions sections).

In [ ]:
# ===== CONFIG =====
PMC_BASE    = "/content/drive/MyDrive/Main Thesis/PMC"
META_CSV    = f"{PMC_BASE}/OG/pmc_test.csv_fix.csv"     # PMC open-access file list (16,142 articles)
RAW_CSV     = f"{PMC_BASE}/New_last.csv"                # fetched articles
CORPUS_CSV  = f"{PMC_BASE}/pmc_corpus.csv"
CHECKPOINT  = f"{PMC_BASE}/pmc_fetch_checkpoint.csv"
FETCH_DELAY = 0.4                                       # seconds between NCBI calls

In [ ]:
!pip -q install pandas tqdm requests
import os, re, json, time, requests, numpy as np, pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
os.makedirs(PMC_BASE, exist_ok=True)

In [ ]:
meta = pd.read_csv(META_CSV, encoding="utf-8-sig").dropna()
pmc_list = meta["PMCID"].astype(str).str.strip().tolist()
print(len(pmc_list), "articles in the OA list")

In [ ]:
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
session = requests.Session()
try:
    from google.colab import userdata
    NCBI_KEY = userdata.get("NCBI_API_KEY")
except Exception:
    NCBI_KEY = None

def fetch_xml(pmc_id, max_retry=3):
    """Download the XML record of one PMC article from NCBI E-utilities (with retries)."""
    params = {"db": "pmc", "id": pmc_id, "retmode": "xml"}
    if NCBI_KEY:
        params["api_key"] = NCBI_KEY
    for _ in range(max_retry):
        try:
            r = session.get(BASE_URL, params=params, timeout=20)
            if r.status_code == 200:
                return ET.fromstring(r.text)
        except Exception:
            pass
        time.sleep(2)
    return None

def _sections(parent, para_xpath):
    """Collect (section title, paragraphs) for every section under an XML element."""
    out = []
    for sec in parent.findall(".//sec"):
        paras = ["".join(p.itertext()).strip() for p in sec.findall(para_xpath)]
        if paras:
            out.append({"section_title": sec.findtext("title"), "paragraphs": paras})
    return out

def parse_abstract(root):
    """Return the abstract as a list of sections; an unstructured abstract becomes one section."""
    abstract = root.find(".//abstract")
    if abstract is None:
        return None
    secs = _sections(abstract, ".//p")
    if not secs:   # unstructured abstract
        secs = [{"section_title": "Abstract",
                 "paragraphs": ["".join(p.itertext()).strip() for p in abstract.findall("./p")]}]
    return secs

def parse_fulltext(root):
    """Return the body of the article as a list of sections."""
    body = root.find(".//body")
    return None if body is None else _sections(body, "./p")

def fetch_article(pmc_id):
    """Fetch one article and return its metadata, abstract and full text (sections stored as JSON)."""
    root = fetch_xml(pmc_id)
    if root is None:
        return None
    lic = root.find(".//license/license-p")
    title = root.find(".//article-title")
    authors = [f"{c.findtext('.//given-names')} {c.findtext('.//surname')}"
               for c in root.findall('.//contrib[@contrib-type="author"]')
               if c.findtext(".//surname") and c.findtext(".//given-names")]
    return {"PMCID": pmc_id,
            "License": "".join(lic.itertext()).strip() if lic is not None else None,
            "Source": f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmc_id}",
            "Journal": root.findtext(".//journal-title"),
            "Title": "".join(title.itertext()).strip() if title is not None else None,
            "Authors": ", ".join(authors),
            "Abstract": json.dumps(parse_abstract(root), ensure_ascii=False),
            "FullText": json.dumps(parse_fulltext(root), ensure_ascii=False)}

def build_dataframe(pmc_ids, delay=FETCH_DELAY, save_every=100):
    """Fetch all articles, saving a checkpoint every save_every articles so the download can resume."""
    done = pd.read_csv(CHECKPOINT) if os.path.exists(CHECKPOINT) else pd.DataFrame()
    seen = set(done["PMCID"]) if len(done) else set()
    records = done.to_dict("records")
    for i, pid in enumerate(tqdm([p for p in pmc_ids if p not in seen], desc="Fetching PMC")):
        rec = fetch_article(pid)
        if rec:
            records.append(rec)
        if (i + 1) % save_every == 0:
            pd.DataFrame(records).to_csv(CHECKPOINT, index=False)
        time.sleep(delay)
    df = pd.DataFrame(records)
    df.to_csv(CHECKPOINT, index=False)
    return df

In [ ]:
raw = build_dataframe(pmc_list)
raw.to_csv(RAW_CSV, index=False)
print(len(raw), "articles fetched")

In [ ]:
def classify_license(text):
    """Map a licence statement to a licence type (CC-BY-x.x, CC0, other CC variants, Other, Unknown)."""
    if pd.isna(text):
        return "Unknown"
    t = text.lower()
    m = re.search(r"creativecommons\.org/licenses/by/(\d\.\d)", t)
    if m:
        return f"CC-BY-{m.group(1)}"
    if "creativecommons.org/publicdomain/zero" in t or "cc0" in t:
        return "CC0"
    for key, name in [("by-nc-sa", "CC-BY-NC-SA"), ("by-nc-nd", "CC-BY-NC-ND"), ("by-nc", "CC-BY-NC"),
                      ("by-sa", "CC-BY-SA"), ("by-nd", "CC-BY-ND")]:
        if key in t:
            return name
    return "Other"

def is_empty(js):
    """True if a stored JSON field is missing or empty."""
    try:
        v = json.loads(js)
    except Exception:
        return True
    return not v

df = raw.copy()
df["LicenseType"] = df["License"].apply(classify_license)
print(df["LicenseType"].value_counts())
df = df[~df["LicenseType"].isin(["Other", "Unknown"])]
df = df[~df["Abstract"].apply(is_empty) & ~df["FullText"].apply(is_empty)]
df = df.dropna(subset=["PMCID", "Title", "Authors", "Journal", "License", "Source"])
print(len(df), "articles after licence / empty filter")

In [ ]:
def clean_section(s):
    """Normalise a section heading: lower case, no trailing punctuation, '&' -> 'and', '/' -> space."""
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[:\.]+$", "", s).replace("&", "and").replace("/", " ")
    return re.sub(r"\s+", " ", s)

BACKGROUND = {"introduction", "background", "backgrounds", "background and purpose", "background and aims",
              "background and objectives", "background context", "summary of background data"}
OBJECTIVE = {"objective", "objectives", "aim", "aims", "aims and objectives", "study aim", "study objective",
             "purpose", "purpose objectives", "purpose of review", "statement of the problem",
             "questions purposes", "study question"}
METHODS = {"methods", "method", "methodology", "methods design", "method design", "methods and design",
           "methods and materials", "methods and patients", "methods materials", "materials and methods",
           "material and methods", "material and method", "materials and method", "patients and methods",
           "patients and method", "patients methods", "subjects and methods", "study design",
           "study design and methods", "design", "design and methods", "design and setting",
           "experimental design", "research design", "research design and methods", "data and methods",
           "study methods", "methodological approach", "participants materials setting methods"}
RESULTS = {"results", "result", "findings", "finding", "methods results", "evaluation results and discussions",
           "measurements and main results", "main results and the role of chance", "recent findings"}
CONCLUSION = {"conclusion", "conclusions", "conclusions and implications", "conclusions impact", "implications",
              "clinical relevance", "wider implications of the findings", "limitations reasons for caution",
              "limitations", "recommendations"}

def classify_section(s):
    """Map a normalised abstract heading to Background, Objective, Methods, Results, Conclusion or Other."""
    s = clean_section(s)
    for name, vocab in [("Background", BACKGROUND), ("Objective", OBJECTIVE), ("Methods", METHODS),
                        ("Results", RESULTS), ("Conclusion", CONCLUSION)]:
        if s in vocab:
            return name
    return "Other"

def extract_sections(abstract_json):
    """Split a structured abstract into its Background, Objective, Methods, Results and Conclusion text."""
    secs = {k: [] for k in ["Background", "Objective", "Methods", "Results", "Conclusion", "Other"]}
    for sec in json.loads(abstract_json) or []:
        secs[classify_section(sec.get("section_title", ""))].append(" ".join(sec.get("paragraphs", [])))
    return pd.Series({k: (" ".join(v) if v else None) for k, v in secs.items()})

df = pd.concat([df.reset_index(drop=True), df["Abstract"].apply(extract_sections).reset_index(drop=True)], axis=1)
df = df.dropna(subset=["Background", "Conclusion"])
print(len(df), "articles with structured Background + Conclusion")

In [ ]:
INTRO_SECTIONS = {"background", "introduction", "overview"}
CONCLUSION_SECTIONS = {"conclusion", "conclusions"}

def extract_intro_conclusion(fulltext_json):
    """Extract the Introduction and Conclusions sections of the full text (accepted heading variants)."""
    intro, concl = [], []
    for sec in json.loads(fulltext_json) or []:
        t = clean_section(sec.get("section_title"))
        text = " ".join(sec.get("paragraphs", []))
        if t in INTRO_SECTIONS:
            intro.append(text)
        elif t in CONCLUSION_SECTIONS:
            concl.append(text)
    return pd.Series({"Introduction": " ".join(intro) or None, "Conclusion_FT": " ".join(concl) or None})

df = pd.concat([df.reset_index(drop=True), df["FullText"].apply(extract_intro_conclusion)], axis=1)
df = df.dropna(subset=["Introduction", "Conclusion_FT"])
corpus = df.rename(columns={"Background": "Target_I", "Conclusion": "Target_C"})
corpus = corpus.rename(columns={"Conclusion_FT": "Conclusion"})
corpus = corpus[["PMCID", "Title", "Authors", "Journal", "License", "LicenseType", "Source",
                 "Abstract", "FullText", "Introduction", "Conclusion", "Target_I", "Target_C"]]
corpus.to_csv(CORPUS_CSV, index=False)
print(len(corpus), "articles saved to", CORPUS_CSV)

## 2.2 Article-level split, records and topic annotation (MedCAT)
Articles are split 70/15/15 into training, validation and test sets **before** any record or pair is built, so no article appears in more than one split. Each training article yields up to two records: Introduction → abstract Background, and Conclusions → abstract Conclusion. Records are kept only if the context fits the token budget (they are filtered, not truncated). Topics are extracted from each target with MedCAT.

`EVAL_CONTEXT` sets the input of the validation and test records. The results reported in the manuscript used `"abstract"`, in which the whole abstract is the input; this makes the biomedical test partly extractive (see Limitations). `"fulltext"` builds them in the same way as the training records.

In [ ]:
# ===== CONFIG =====
MEDCAT_PACK      = "/content/drive/MyDrive/medmen_wstatus_2021_oct"   # MedCAT model pack (MedMentions)
SPLIT_SEED       = 14
INTRO_MAX_TOKENS = 500
CONC_MAX_TOKENS  = 150
EVAL_CONTEXT     = "abstract"        # "abstract" (as reported) | "fulltext"

In [ ]:
!pip -q install medcat==2.6.0
!python -m spacy download en_core_web_md -q
from sklearn.model_selection import train_test_split
corpus = pd.read_csv(CORPUS_CSV)
train_art, temp = train_test_split(corpus, test_size=0.3, random_state=SPLIT_SEED)
val_art, test_art = train_test_split(temp, test_size=0.5, random_state=SPLIT_SEED)
print(f"articles: train {len(train_art)} | val {len(val_art)} | test {len(test_art)}")
assert not (set(train_art.PMCID) & set(test_art.PMCID)) and not (set(val_art.PMCID) & set(test_art.PMCID))

In [ ]:
KEEP = ["PMCID", "Title", "Authors", "Journal", "License", "LicenseType", "Source"]

def ntok(x):
    """Number of T5 tokens in a text (None for missing text)."""
    return len(tokenizer(x, add_special_tokens=False)["input_ids"]) if pd.notna(x) else None

def make_records(art):
    """Full-text records: Introduction -> Background and Conclusions -> Conclusion, kept only if the
    context is within the token budget (INTRO_MAX_TOKENS / CONC_MAX_TOKENS)."""
    rows = []
    for kind, ctx_col, tgt_col, limit in [("Introduction", "Introduction", "Target_I", INTRO_MAX_TOKENS),
                                          ("Conclusion", "Conclusion", "Target_C", CONC_MAX_TOKENS)]:
        r = art[KEEP + [ctx_col, tgt_col]].rename(columns={ctx_col: "Context", tgt_col: "Target"})
        r["Type"] = kind
        r["token"] = r["Context"].apply(ntok)
        rows.append(r[r["token"] <= limit])
    return pd.concat(rows, ignore_index=True)

def abstract_text(abstract_json):
    """Concatenate all paragraphs of a stored abstract into one text."""
    secs = json.loads(abstract_json) or []
    return " ".join(" ".join(s.get("paragraphs", [])) for s in secs)

def make_abstract_records(art):
    """Evaluation records whose input is the whole abstract (two records per article)."""
    rows = []
    for kind, tgt_col in [("Introduction", "Target_I"), ("Conclusion", "Target_C")]:
        r = art[KEEP].copy()
        r["Context"] = art["Abstract"].apply(abstract_text).values
        r["Target"] = art[tgt_col].values
        r["Type"] = kind
        rows.append(r)
    return pd.concat(rows, ignore_index=True).sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

make_eval = make_abstract_records if EVAL_CONTEXT == "abstract" else make_records
recs = {"train": make_records(train_art), "val": make_eval(val_art), "test": make_eval(test_art)}
for k, v in recs.items():
    print(k, len(v), "records |", v["Type"].value_counts().to_dict())

In [ ]:
from medcat.cat import CAT
cat = CAT.load_model_pack(MEDCAT_PACK)

def extract_topic(texts):
    """Topic of each target: the MedCAT concept with the highest accuracy score."""
    out = []
    for text in tqdm(texts, desc="MedCAT"):
        if pd.isna(text):
            out.append(None); continue
        ents = cat.get_entities(text)["entities"]
        if not ents:
            out.append(None); continue
        out.append(max(ents.values(), key=lambda e: e.get("acc", 0))["source_value"])
    return out

for k in recs:
    recs[k]["topic"] = extract_topic(recs[k]["Target"])
    recs[k] = recs[k].dropna(subset=["topic"]).reset_index(drop=True)

# Articles whose Introduction and Conclusion topics are identical are removed from validation and test,
# because the two topics could not serve as each other's alternative.
for k in ["val", "test"]:
    r = recs[k]
    piv = r.pivot_table(index="PMCID", columns="Type", values="topic", aggfunc="first")
    same = piv.index[(piv.get("Introduction").str.lower() == piv.get("Conclusion").str.lower()).fillna(False)]
    recs[k] = r[~r["PMCID"].isin(same)].reset_index(drop=True)

for k, v in recs.items():
    v.to_csv(f"{PMC_BASE}/pmc+_{k}.csv", index=False)
    print(k, len(v), "records |", v.PMCID.nunique(), "articles")

## 2.3 Pair construction (ψ), difficulty tiers and augmentation scales (ξ)
Pairs of training records are scored by the cosine similarity of their targets and kept only if both contexts together fit within 500 tokens. Each pair has a type (II, IC or CC, by the section of each context). Scales 1X–30X are balanced across the nine tier × type cells as far as each cell allows.

In [ ]:
# ===== CONFIG =====
POOL_SIZE       = 100_000
MAX_PAIR_TOKENS = 500
PMC_TIERS       = (0.0979, 0.2147)       # similarity thresholds between tiers 1/2 and 2/3 (training pools)
EVAL_TIERS      = (0.10, 0.21)           # thresholds used to stratify the validation and test pairs
STEP            = 2000
SCALES          = list(range(1, 17)) + [25, 30]
TIER_SET_SIZE   = 28_000                 # size of the tier-configuration sets (RQ3) = 14X
TIER_TYPE_RATIO = (49, 44)               # IC : CC ratio after all II pairs in the tier sets
VAL_SIZE, TEST_SIZE = 1500, 1300

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("all-MiniLM-L6-v2")
recs = {k: pd.read_csv(f"{PMC_BASE}/pmc+_{k}.csv") for k in ["train", "val", "test"]}
TYPE_CODE = {"Introduction": "I", "Conclusion": "C"}

def build_pairs(df, max_pairs, seed, max_tokens=MAX_PAIR_TOKENS):
    """Draw up to max_pairs record pairs whose contexts fit together in max_tokens, scored by the
    cosine similarity of their targets. Returns one mixed-context row per pair."""
    rng = np.random.default_rng(seed)
    df = df.reset_index(drop=True)
    emb = st_model.encode(df["Target"].tolist(), batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    sim = emb @ emb.T
    tok = np.array([len(tokenizer.encode(x)) for x in df["Context"]])
    ok = (tok[:, None] + tok[None, :] <= max_tokens) & (sim >= 0)
    np.fill_diagonal(ok, False)
    cand = np.flatnonzero(ok)
    a, b = np.unravel_index(rng.choice(cand, size=min(max_pairs, len(cand)), replace=False), ok.shape)
    A, B = df.iloc[a].reset_index(drop=True), df.iloc[b].reset_index(drop=True)
    return pd.DataFrame({
        "PMCID1": A["PMCID"], "PMCID2": B["PMCID"],
        "Combined_text": A["Context"] + " " + B["Context"],
        "Context1": A["Context"], "Context2": B["Context"],
        "Target1": A["Target"], "Target2": B["Target"],
        "Topic1": A["topic"], "Topic2": B["topic"],
        "Type": (A["Type"].map(TYPE_CODE) + B["Type"].map(TYPE_CODE)).replace({"CI": "IC"}),
        "similarity_score": sim[a, b]})

def assign_tier(scores, th):
    """Map similarity scores to difficulty tiers 1 (low), 2 (medium) and 3 (high)."""
    return np.where(scores < th[0], 1, np.where(scores <= th[1], 2, 3))

pools = {}
for tag, seed in SEED_MAP.items():
    p = build_pairs(recs["train"], POOL_SIZE, seed)
    p["Terciles"] = assign_tier(p["similarity_score"].values, PMC_TIERS)
    pools[tag] = p
    p.to_csv(f"{PMC_BASE}/pool_100k_{tag}.csv", index=False)
    print(tag, len(p), "pairs | tier %", (p["Terciles"].value_counts(normalize=True).sort_index() * 100).round(1).to_dict())

In [ ]:
def assign_target(df):
    """Choose which context of each pair supplies the target: the first context for the first half of
    the rows and the second context for the second half."""
    df = df.copy()
    swap = np.arange(len(df)) >= len(df) // 2
    for a, b in [("Target1", "Target2"), ("Topic1", "Topic2"), ("PMCID1", "PMCID2")]:
        df.loc[swap, [a, b]] = df.loc[swap, [b, a]].values
    return df

def create_scaled_datasets(df, seed):
    """Build the 1X-30X sets: an equal share from each tier x type cell, topped up from the rest of the
    pool when a cell runs out. The tier shares of each scale are printed."""
    rng = np.random.default_rng(seed)
    groups = {k: rng.permutation(g.index.values) for k, g in df.groupby(["Terciles", "Type"])}
    out = {}
    for k in SCALES:
        target = k * STEP
        chosen = np.concatenate([idx[:target // len(groups)] for idx in groups.values()])
        need = target - len(chosen)
        if need > 0:
            rest = df.index.difference(chosen)
            chosen = np.concatenate([chosen, rng.choice(rest.values, size=need, replace=False)])
        d = assign_target(df.loc[chosen].reset_index(drop=True))
        print(f"{k:>2}X n={len(d):>6} tier% {(d['Terciles'].value_counts(normalize=True).sort_index() * 100).round(1).to_dict()}")
        out[f"{k}x"] = d
    return out

scaled = {}
for tag, p in pools.items():
    scaled[tag] = create_scaled_datasets(p, SEED_MAP[tag])
    os.makedirs(f"{PMC_BASE}/Mix_PMC+_{tag}/Mixed/Data", exist_ok=True)
    for k, d in scaled[tag].items():
        d.to_csv(f"{PMC_BASE}/Mix_PMC+_{tag}/Mixed/Data/PMC{k}_{tag}.csv", index=False)

In [ ]:
def tier_set(df, tiers, seed, total=TIER_SET_SIZE, ratio=TIER_TYPE_RATIO):
    """Tier-configuration set for RQ3: an equal share from each selected tier; within a tier all II pairs
    are taken first and the remainder is split between IC and CC pairs in the given ratio."""
    parts, per_tier = [], total // len(tiers)
    for t in tiers:
        d = df[df["Terciles"] == t]
        ii = d[d["Type"] == "II"]
        if len(ii) > per_tier:
            ii = ii.sample(per_tier, random_state=seed)
        rem = per_tier - len(ii)
        n_ic = int(rem * ratio[0] / sum(ratio))
        ic, cc = d[d["Type"] == "IC"], d[d["Type"] == "CC"]
        parts += [ii, ic.sample(min(n_ic, len(ic)), random_state=seed), cc.sample(min(rem - n_ic, len(cc)), random_state=seed)]
    return assign_target(pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True))

TIER_SETS = {"T1": [1], "T2": [2], "T3": [3], "T1T2": [1, 2], "T1T3": [1, 3], "T2T3": [2, 3]}
for tag, p in pools.items():
    os.makedirs(f"{PMC_BASE}/tiers_{tag}", exist_ok=True)
    for name, tiers in TIER_SETS.items():
        d = tier_set(p, tiers, SEED_MAP[tag])
        d.to_csv(f"{PMC_BASE}/tiers_{tag}/{name}_{tag}.csv", index=False)
        print(tag, name, len(d), d["Type"].value_counts().to_dict())

In [ ]:
def proportional_sample(df, target, seed):
    """Sample `target` pairs keeping the tier x type proportions of the candidate pool."""
    rng = np.random.default_rng(seed)
    parts = [g.loc[rng.permutation(g.index.values)[:min(len(g), round(len(g) / len(df) * target))]]
             for _, g in df.groupby(["Terciles", "Type"])]
    out = pd.concat(parts)
    if len(out) < target:
        out = pd.concat([out, df.drop(out.index).sample(target - len(out), random_state=seed)])
    return out.reset_index(drop=True)

for split, size, name in [("val", VAL_SIZE, "PMC+_val.csv"), ("test", TEST_SIZE, "PMC+_test.csv")]:
    pairs = build_pairs(recs[split], 100_000, 14, max_tokens=10**6 if EVAL_CONTEXT == "abstract" else MAX_PAIR_TOKENS)
    pairs["Terciles"] = assign_tier(pairs["similarity_score"].values, EVAL_TIERS)
    d = assign_target(proportional_sample(pairs, size, 14))
    d.to_csv(f"{PMC_BASE}/{name}", index=False)
    print(split, len(d), "pairs |", pd.unique(pd.concat([d.PMCID1, d.PMCID2])).size, "articles")

In [ ]:
for tag in scaled:
    for k, d in scaled[tag].items():
        to_pth(d, "Topic1", "Combined_text", "Target1", f"{PMC_BASE}/Mix_PMC+_{tag}/Mixed/Data/{k}.pth")
    for name in TIER_SETS:
        d = pd.read_csv(f"{PMC_BASE}/tiers_{tag}/{name}_{tag}.csv")
        to_pth(d, "Topic1", "Combined_text", "Target1", f"{PMC_BASE}/tiers_{tag}/{name}.pth")
to_pth(pd.read_csv(f"{PMC_BASE}/PMC+_val.csv"), "Topic1", "Combined_text", "Target1", f"{PMC_BASE}/val.pth")
to_pth(recs["train"], "topic", "Context", "Target", f"{PMC_BASE}/baseline1_train.pth")   # Baseline1: no mixing

## 2.4 Training

In [ ]:
PMC_VAL_PTH = f"{PMC_BASE}/val.pth"
PMC_RUNS = {f"{k}x": lambda tag, k=k: f"{PMC_BASE}/Mix_PMC+_{tag}/Mixed/Data/{k}x.pth" for k in SCALES}
PMC_RUNS.update({name: lambda tag, n=name: f"{PMC_BASE}/tiers_{tag}/{n}.pth" for name in TIER_SETS})
PMC_RUNS["baseline1"] = lambda tag: f"{PMC_BASE}/baseline1_train.pth"

for tag, seed in SEED_MAP.items():
    for name, pth in PMC_RUNS.items():
        train_t5(pth(tag), PMC_VAL_PTH, f"{PMC_BASE}/models/{name}_{tag}", seed)

## 2.5 Generating test summaries

In [ ]:
PMC_TEST_CSV = f"{PMC_BASE}/PMC+_test.csv"
def pmc_result_path(name, seed):
    """Location of the generated summaries of one model and seed."""
    return f"{PMC_BASE}/Mix_PMC+_{seed}/Mixed/Result/pmc_{name}_{seed}.csv"

ptest = pd.read_csv(PMC_TEST_CSV)
for tag in SEED_MAP:
    for name in PMC_RUNS:
        generate_summaries(f"{PMC_BASE}/models/{name}_{tag}/best_model.pth", ptest["Topic1"], ptest["Combined_text"],
                           ptest["Target1"], pmc_result_path(name, tag))

## 2.6 Evaluation
Same metrics as the science domain. The alternative topic of each test pair is the topic of the other article in the pair (`Topic2`), so the chance level of the Win Rate is 50%.

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from transformers.utils import logging
logging.set_verbosity_error()
SEEDS = ["000", "100", "200"]
st = SentenceTransformer("all-MiniLM-L6-v2")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
ptest = pd.read_csv(PMC_TEST_CSV); ptest.columns = [c.lower() for c in ptest.columns]
REF_T, ALT_T = ptest["topic1"].astype(str).tolist(), ptest["topic2"].astype(str).tolist()
VOCAB = sorted(set(REF_T) | set(ALT_T))
TV = dict(zip(VOCAB, st.encode(VOCAB, convert_to_numpy=True, normalize_embeddings=True)))
print(len(ptest), "test rows")

def score_file_pmc(preds, refs):
    """Summary-quality and topic-alignment metrics for one biomedical result file (rows in test-set order)."""
    preds = [str(p) if pd.notna(p) else "" for p in preds]
    refs  = [str(r) for r in refs]
    rl = np.mean([scorer.score(p, r)["rougeL"].fmeasure for p, r in zip(preds, refs)])
    _, _, f1 = bert_score(preds, refs, lang="en", verbose=False)
    pe = st.encode(preds, convert_to_numpy=True, normalize_embeddings=True)
    re_ = st.encode(refs, convert_to_numpy=True, normalize_embeddings=True)
    ref = np.array([pe[i] @ TV[t] for i, t in enumerate(REF_T)])
    alt = np.array([pe[i] @ TV[t] for i, t in enumerate(ALT_T)])
    win = (ref > alt).astype(float)
    return {"rougeL": rl, "bert_f1": f1.mean().item(), "cosine_sim": float((pe * re_).sum(1).mean()),
            "wins": int(win.sum()), "win_rate": 100 * win.mean(),
            "avg_ref": float(ref.mean()), "avg_alt": float(alt.mean()), "diff": float(ref.mean() - alt.mean())}

def run_seed_pmc(seed, names=None):
    """Score every model of one seed and save the per-seed table."""
    rows = []
    for name in tqdm(names or [f"{n}x" for n in SCALES], desc=f"seed {seed}"):
        path = pmc_result_path(name, seed)
        if not os.path.exists(path):
            print("not found:", path); continue
        d = pd.read_csv(path)
        if len(d) != len(ptest):
            print(f"skipped {path}: {len(d)} rows"); continue
        rows.append({"model": name, "seed": seed, **score_file_pmc(d["prediction"], d["reference"])})
    df = pd.DataFrame(rows)
    df.to_csv(f"{PMC_BASE}/per_seed_pmc_{seed}.csv", index=False)
    return df

pmc_per_seed = pd.concat([run_seed_pmc(s) for s in SEEDS], ignore_index=True)
pmc_per_seed.to_csv(f"{PMC_BASE}/per_seed_pmc.csv", index=False)
pmc_per_seed.groupby("model").mean(numeric_only=True).round(4)

## 2.7 Prompted LLM baselines

In [ ]:
PMC_LLM_DIR = f"{PMC_BASE}/LLM-0-shot"
os.makedirs(PMC_LLM_DIR, exist_ok=True)
ptest_raw = pd.read_csv(PMC_TEST_CSV)
llm_rows = []
for model_id in PROVIDERS:
    out = run_llm(model_id, ptest_raw, "Topic1", "Combined_text", f"{PMC_LLM_DIR}/llm_raw_{model_id.replace('/', '_')}.csv")
    out.to_csv(f"{PMC_LLM_DIR}/llm_{model_id.split('/')[-1]}.csv", index=False)
    llm_rows.append({"model": model_id, **score_file_pmc(out["prediction"], out["Target1"])})
pd.DataFrame(llm_rows).round(4)

# Part 3 · Additional experiments (revision)
These analyses were added in response to the reviewers. They reuse the trained models and result files from Parts 1 and 2.

## 3.1 Re-evaluation with a second encoder
All result files are re-scored with Qwen3-Embedding-0.6B, which plays no role in tier construction. The instruction prompt is placed on the topic side. Only the topic-alignment metrics are recomputed; the first row of each table is the score of the human-written reference summaries.

In [ ]:
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
login(token=userdata.get("HF_KEY"))
QWEN = "Qwen/Qwen3-Embedding-0.6B"

def make_variants(domain_word):
    """Encoder settings: all-MiniLM-L6-v2 (as in the main evaluation) and Qwen3 with the prompt on the topic side."""
    return {"minilm": {"id": "sentence-transformers/all-MiniLM-L6-v2", "dtype": torch.float32, "pred_prompt": None, "topic_prompt": None},
            "qwen3":  {"id": QWEN, "dtype": torch.float16, "pred_prompt": None,
                       "topic_prompt": f"Instruct: Given a topic, retrieve a summary of a {domain_word} about it\nQuery: "}}

def encode_with(model, cfg, texts, side):
    """Encode texts with the prompt of the given side ("topic" or "pred")."""
    return model.encode([str(t) if pd.notna(t) else "" for t in texts], prompt=cfg[f"{side}_prompt"], batch_size=64,
                        convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

def summarise(df, order):
    """Mean and standard deviation over seeds of the topic-alignment metrics, in scale order."""
    return (df.groupby("model")[["win_rate", "avg_ref", "avg_alt", "diff"]].agg(["mean", "std"])
              .reindex([m for m in order if m in set(df.model)]).round(4))

### Science domain

In [ ]:
test = pd.read_csv(SCI_TEST_CSV)
ALT = test["alter_topic_filtered"].apply(lambda x: [str(t) for t in ast.literal_eval(str(x))] if str(x).startswith("[") else [])
SCI_VOCAB = sorted({str(t) for t in test["topic"]} | {t for a in ALT for t in a})

def sci_topic_scores(model, cfg, preds):
    """Win Rate, Avg_ref, Avg_alt and Diff of one result file under a given encoder (science domain)."""
    tv = dict(zip(SCI_VOCAB, encode_with(model, cfg, SCI_VOCAB, "topic")))
    pe = encode_with(model, cfg, preds, "pred")
    ref = np.array([pe[i] @ tv[str(t)] for i, t in enumerate(test["topic"])])
    win, alt = [], []
    for i, a in enumerate(ALT):
        if not a:
            win.append(1.0); alt.append(np.nan); continue
        s = np.array([pe[i] @ tv[t] for t in a]); win.append(float(ref[i] > s.max())); alt.append(s.mean())
    return {"win_rate": 100 * np.mean(win), "avg_ref": float(ref.mean()), "avg_alt": float(np.nanmean(alt)),
            "diff": float(ref.mean() - np.nanmean(alt))}

def run_encoder_sci(name, variants):
    """Re-score the reference summaries and every science result file with one encoder setting."""
    cfg = variants[name]
    model = SentenceTransformer(cfg["id"], device="cuda", model_kwargs={"torch_dtype": cfg["dtype"]})
    rows = [{"variant": name, "model": "reference", "seed": "-", **sci_topic_scores(model, cfg, test["target"])}]
    for seed in SEEDS:
        for n in tqdm(SCALES, desc=f"{name} {seed}"):
            p = sci_result_path(f"{n}x", seed)
            if os.path.exists(p):
                rows.append({"variant": name, "model": f"{n}x", "seed": seed,
                             **sci_topic_scores(model, cfg, pd.read_csv(p)["prediction"])})
    del model; gc.collect(); torch.cuda.empty_cache()
    df = pd.DataFrame(rows); df.to_csv(f"{SCI_BASE}/per_seed_{name}.csv", index=False)
    return df

SCI_VARIANTS = make_variants("scientific paper")
sci_enc = pd.concat([run_encoder_sci(v, SCI_VARIANTS) for v in SCI_VARIANTS], ignore_index=True)
sci_enc.to_csv(f"{SCI_BASE}/per_seed_all_encoders.csv", index=False)
for v in SCI_VARIANTS:
    display(summarise(sci_enc[sci_enc.variant == v], ["reference"] + [f"{n}x" for n in SCALES]))

### Biomedical domain
Result files must be in the same row order as the test file. Check that the Win Rate of every seed is well above the 50% chance level before pooling seeds.

In [ ]:
ptest = pd.read_csv(PMC_TEST_CSV); ptest.columns = [c.lower() for c in ptest.columns]
REF_T, ALT_T = ptest["topic1"].astype(str).tolist(), ptest["topic2"].astype(str).tolist()
PMC_VOCAB = sorted(set(REF_T) | set(ALT_T))

def pmc_topic_scores(model, cfg, preds):
    """Win Rate, Avg_ref, Avg_alt and Diff of one result file under a given encoder (biomedical domain)."""
    tv = dict(zip(PMC_VOCAB, encode_with(model, cfg, PMC_VOCAB, "topic")))
    pe = encode_with(model, cfg, preds, "pred")
    ref = np.array([pe[i] @ tv[t] for i, t in enumerate(REF_T)])
    alt = np.array([pe[i] @ tv[t] for i, t in enumerate(ALT_T)])
    return {"win_rate": 100 * (ref > alt).mean(), "avg_ref": float(ref.mean()), "avg_alt": float(alt.mean()),
            "diff": float(ref.mean() - alt.mean())}

def run_encoder_pmc(name, variants):
    """Re-score the reference summaries and every biomedical result file with one encoder setting."""
    cfg = variants[name]
    model = SentenceTransformer(cfg["id"], device="cuda", model_kwargs={"torch_dtype": cfg["dtype"]})
    rows = [{"variant": name, "model": "reference", "seed": "-", **pmc_topic_scores(model, cfg, ptest["target1"])}]
    for seed in SEEDS:
        for n in tqdm(SCALES, desc=f"{name} {seed}"):
            p = pmc_result_path(f"{n}x", seed)
            if os.path.exists(p):
                rows.append({"variant": name, "model": f"{n}x", "seed": seed,
                             **pmc_topic_scores(model, cfg, pd.read_csv(p)["prediction"])})
    del model; gc.collect(); torch.cuda.empty_cache()
    df = pd.DataFrame(rows); df.to_csv(f"{PMC_BASE}/per_seed_pmc_{name}.csv", index=False)
    return df

PMC_VARIANTS = make_variants("research article")
pmc_enc = pd.concat([run_encoder_pmc(v, PMC_VARIANTS) for v in PMC_VARIANTS], ignore_index=True)
pmc_enc.to_csv(f"{PMC_BASE}/per_seed_pmc_all_encoders.csv", index=False)
print(pmc_enc[pmc_enc.model != "reference"].groupby(["variant", "seed"])["win_rate"].mean().round(2))
for v in PMC_VARIANTS:
    display(summarise(pmc_enc[pmc_enc.variant == v], ["reference"] + [f"{n}x" for n in SCALES]))

## 3.2 Size-matched random-mixing baseline
Pairs are drawn uniformly from the same pool as the balanced model, without balancing the tiers, with the same number of pairs: 14,000 (7X) for science and 28,000 (14X) for biomedicine. Models are trained and evaluated with the same settings and three seeds.

In [ ]:
def load_lower(path):
    """Read a CSV and lower-case its column names (the science and biomedical files differ in case)."""
    d = pd.read_csv(path); d.columns = [c.lower() for c in d.columns]; return d

def random_mix(pool, n, seed):
    """Draw n pairs uniformly from the pool, ignoring tiers, and take the target from either context
    with equal probability."""
    rng = np.random.default_rng(seed)
    d = pool.loc[rng.choice(pool.index.values, size=n, replace=False)].reset_index(drop=True)
    swap = rng.random(len(d)) < 0.5
    for a, b in [("target1", "target2"), ("topic1", "topic2"), ("pmcid1", "pmcid2")]:
        if a in d.columns and b in d.columns:
            d.loc[swap, [a, b]] = d.loc[swap, [b, a]].values
    return d

def tier_shares(df):
    """Percentage of pairs in each tier."""
    return (df["terciles"].value_counts(normalize=True).sort_index() * 100).round(1).to_dict()

RANDOM_SETUP = {   # domain: (base folder, scale, tier thresholds, validation .pth)
    "science":    (SCI_BASE, 7, SCI_TIERS, f"{SCI_BASE}/real_val.pth"),
    "biomedical": (PMC_BASE, 14, PMC_TIERS, f"{PMC_BASE}/val.pth"),
}
for domain, (base, k, th, val_pth) in RANDOM_SETUP.items():
    for tag, seed in SEED_MAP.items():
        pool = load_lower(f"{base}/pool_100k_{tag}.csv").drop_duplicates(["context1", "context2"]).reset_index(drop=True)
        pool["terciles"] = assign_tier(pool["similarity_score"].values, th)
        d = random_mix(pool, k * STEP, seed + 1000)
        os.makedirs(f"{base}/random_{tag}", exist_ok=True)
        d.to_csv(f"{base}/random_{tag}/Random_{k}x_{tag}.csv", index=False)
        print(domain, tag, len(d), "pairs | tier %", tier_shares(d))
        to_pth(d, "topic1", "combined_text", "target1", f"{base}/random_{tag}/Random_{k}x.pth")
        train_t5(f"{base}/random_{tag}/Random_{k}x.pth", val_pth, f"{base}/models/random{k}x_{tag}", seed)

In [ ]:
# Generate test summaries and score the random baselines next to the balanced models
test = pd.read_csv(SCI_TEST_CSV); ptest_raw = pd.read_csv(PMC_TEST_CSV)
rows = []
for tag in SEED_MAP:
    out = f"{SCI_BASE}/random_{tag}/Result/result7x_random_{tag}.csv"
    generate_summaries(f"{SCI_BASE}/models/random7x_{tag}/best_model.pth", test["topic"], test["source"], test["target"], out)
    rows.append({"domain": "science", "model": "Random7X", "seed": tag, **score_file(pd.read_csv(out)["prediction"], pd.read_csv(out)["reference"])})
    d = pd.read_csv(sci_result_path("7x", tag))
    rows.append({"domain": "science", "model": "TCS7X", "seed": tag, **score_file(d["prediction"], d["reference"])})
    out = f"{PMC_BASE}/random_{tag}/Result/result14x_random_{tag}.csv"
    generate_summaries(f"{PMC_BASE}/models/random14x_{tag}/best_model.pth", ptest_raw["Topic1"], ptest_raw["Combined_text"], ptest_raw["Target1"], out)
    rows.append({"domain": "biomedical", "model": "Random14X", "seed": tag, **score_file_pmc(pd.read_csv(out)["prediction"], pd.read_csv(out)["reference"])})
    d = pd.read_csv(pmc_result_path("14x", tag))
    rows.append({"domain": "biomedical", "model": "TCSM14X", "seed": tag, **score_file_pmc(d["prediction"], d["reference"])})
random_vs_balanced = pd.DataFrame(rows)
random_vs_balanced.to_csv(f"{PMC_BASE}/per_seed_random_vs_balanced.csv", index=False)
random_vs_balanced.groupby(["domain", "model"]).agg(["mean", "std"]).round(4)

## 3.3 Statistical tests
* **Random versus balanced:** Welch's two-sample *t*-test on the seed-level values, with the 95% confidence interval of the difference.
* **Scale trend:** Jonckheere–Terpstra test for an ordered increase across the 18 scales, seeds as replicates (normal approximation with tie correction).
* **Agreement between encoders:** Spearman correlation of the scale means, with a bootstrap interval over scales.

In [ ]:
from scipy import stats

def welch(random_vals, balanced_vals):
    """Welch t-test of balanced minus random; returns the means, SDs, difference, its 95% CI, t, df and p."""
    a, b = np.asarray(random_vals, float), np.asarray(balanced_vals, float)
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    diff, se = b.mean() - a.mean(), np.sqrt(va + vb)
    t = stats.ttest_ind(b, a, equal_var=False)
    half = stats.t.ppf(0.975, df) * se
    return {"random": f"{a.mean():.4f} ± {a.std(ddof=1):.4f}", "balanced": f"{b.mean():.4f} ± {b.std(ddof=1):.4f}",
            "difference": round(diff, 4), "ci_low": round(diff - half, 4), "ci_high": round(diff + half, 4),
            "t": round(t.statistic, 2), "df": round(df, 1), "p": t.pvalue}

def jonckheere(groups):
    """Jonckheere-Terpstra test for an increasing trend across ordered groups (two-sided p)."""
    J = sum((y > x) + 0.5 * (y == x) for i in range(len(groups)) for j in range(i + 1, len(groups))
            for x in groups[i] for y in groups[j])
    n = np.array([len(g) for g in groups]); N = n.sum()
    _, ties = np.unique(np.concatenate(groups), return_counts=True)
    mean = (N ** 2 - (n ** 2).sum()) / 4
    var = (N ** 2 * (2 * N + 3) - (n ** 2 * (2 * n + 3)).sum() - (ties * (ties - 1) * (2 * ties + 5)).sum()) / 72
    z = (J - mean) / np.sqrt(var)
    return {"J": J, "z": round(z, 2), "p": 2 * stats.norm.sf(abs(z))}

def spearman_ci(x, y, n_boot=5000, seed=1):
    """Spearman correlation with a 95% bootstrap interval over the paired points."""
    x, y = np.asarray(x), np.asarray(y); rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):
        i = rng.choice(len(x), len(x))
        if len(set(x[i])) > 1 and len(set(y[i])) > 1:
            boots.append(stats.spearmanr(x[i], y[i]).statistic)
    r = stats.spearmanr(x, y)
    return {"rho": round(r.statistic, 3), "p": r.pvalue, "ci": tuple(np.round(np.percentile(boots, [2.5, 97.5]), 2))}

In [ ]:
# Random versus balanced
for domain, (rand_name, bal_name) in {"science": ("Random7X", "TCS7X"), "biomedical": ("Random14X", "TCSM14X")}.items():
    d = random_vs_balanced[random_vs_balanced.domain == domain]
    for metric in ["win_rate", "diff", "rougeL", "bert_f1"]:
        print(domain, metric, welch(d[d.model == rand_name][metric], d[d.model == bal_name][metric]))

In [ ]:
# Scale trend under the second encoder and agreement with all-MiniLM-L6-v2
EXCLUDE_SEEDS = {"science": [], "biomedical": []}   # list seed tags whose result files are misaligned (Win Rate near chance)
for domain, enc in {"science": sci_enc, "biomedical": pmc_enc}.items():
    e = enc[(enc.model != "reference") & (~enc.seed.isin(EXCLUDE_SEEDS[domain]))].copy()
    e["scale"] = e.model.str.rstrip("x").astype(int)
    for metric in ["win_rate", "diff"]:
        q = e[e.variant == "qwen3"]
        groups = [q[q.scale == s][metric].values for s in sorted(q.scale.unique())]
        means = e.groupby(["variant", "scale"])[metric].mean().unstack("variant")
        print(domain, metric, "| JT (Qwen3):", jonckheere(groups),
              "| Spearman MiniLM vs Qwen3:", spearman_ci(means["minilm"], means["qwen3"]),
              "| best scale:", means["minilm"].idxmax(), "/", means["qwen3"].idxmax())

## 3.4 Figure: random versus balanced mixing (means over three seeds)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ['rougeL', 'bert_f1', 'cosine_sim', 'win_rate', 'avg_ref', 'avg_alt', 'diff']

# Science (Win Rate divided by 100 to share the 0-1 scale)
tldr_random7x = [0.2415, 0.8803, 0.5864, 36.6905 / 100, 0.2977, 0.2796, 0.0181]
tldr_tcs7x    = [0.2416, 0.8787, 0.5888, 38.78 / 100,   0.2998, 0.2742, 0.0255]

# Biomedical
pmc_random14x = [0.4541, 0.9011, 0.6274, 71.3077 / 100, 0.2675, 0.1095, 0.1580]
pmc_tcsm14x   = [0.6132, 0.9319, 0.7873, 81.77 / 100,   0.3152, 0.0785, 0.2368]

# One row, two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# --- Panel 1: science ---
ax1.plot(metrics, tldr_random7x, marker='o', linewidth=2, label='Random7X', color='#1f77b4')
ax1.plot(metrics, tldr_tcs7x, marker='s', linewidth=2, linestyle='--', label='TCS7X', color='#ff7f0e')
ax1.set_title('TLDR+ Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score (0 - 1 Scale)', fontsize=11)
ax1.set_ylim(0, 1.05)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(fontsize=11)

# Value labels
for i, (v1, v2) in enumerate(zip(tldr_random7x, tldr_tcs7x)):
    ax1.text(i, v1 - 0.045, f'{v1:.3f}', ha='center', fontsize=8.5, color='#1f77b4')
    ax1.text(i, v2 + 0.025, f'{v2:.3f}', ha='center', fontsize=8.5, color='#ff7f0e')

# --- Panel 2: biomedical ---
ax2.plot(metrics, pmc_random14x, marker='o', linewidth=2, label='Random14X', color='#2ca02c')
ax2.plot(metrics, pmc_tcsm14x, marker='s', linewidth=2, linestyle='--', label='TCSM14X', color='#d62728')
ax2.set_title('PMC+ Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('Score (0 - 1 Scale)', fontsize=11)
ax2.set_ylim(0, 1.05)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(fontsize=11)

# Value labels
for i, (v1, v2) in enumerate(zip(pmc_random14x, pmc_tcsm14x)):
    offset1 = -0.045 if v1 < v2 else 0.025
    offset2 = 0.025 if v1 < v2 else -0.045
    ax2.text(i, v1 + offset1, f'{v1:.3f}', ha='center', fontsize=8.5, color='#2ca02c')
    ax2.text(i, v2 + offset2, f'{v2:.3f}', ha='center', fontsize=8.5, color='#d62728')

plt.tight_layout()
plt.savefig('comparison_charts.png', dpi=300)
plt.show()